# Module 1 - Speaker Attribution Validation

This notebook compares `base` vs `small` for speaker attribution quality on `edinburghcstr/ami` (`ihm`).

Important note:
- `pyannote` speaker labels are anonymous, so direct label equality with `speaker_id` is not meaningful.
- Instead we evaluate proxy metrics that matter for alignment quality:
  - unknown-speaker rate
  - single-speaker diarization rate on single-utterance clips
  - overlap coverage between ASR segments and diarization segments
  - proportion of clips with strong dominant-speaker coverage

In [1]:
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd
import soundfile as sf
from datasets import load_dataset
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / 'backend' / '.env', override=False)

from backend.app.module1 import asr as asr_module
from backend.app.module1 import diarization as diarization_module
from backend.app.module1.alignment import assign_speakers_to_segments

In [2]:
SAMPLE_SIZE = 250
STRONG_COVERAGE_THRESHOLD = 0.8

ds = load_dataset('edinburghcstr/ami', 'ihm', split='test')
sample = ds.shuffle(seed=42).select(range(SAMPLE_SIZE))
sample

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Dataset({
    features: ['meeting_id', 'audio_id', 'text', 'audio', 'begin_time', 'end_time', 'microphone_id', 'speaker_id'],
    num_rows: 250
})

In [3]:
def overlap_duration(start_a: float, end_a: float, start_b: float, end_b: float) -> float:
    return max(0.0, min(end_a, end_b) - max(start_a, start_b))


def evaluate_model(model_size: str) -> pd.DataFrame:
    os.environ['WHISPER_MODEL_SIZE'] = model_size
    os.environ['DIARIZATION_ENABLED'] = 'true'
    asr_module._get_whisper_model.cache_clear()
    diarization_module._get_diarization_pipeline.cache_clear()

    rows = []
    for row in sample:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            sf.write(tmp.name, row['audio']['array'], row['audio']['sampling_rate'])
            asr_result = asr_module.transcribe_audio(tmp.name, chunk_start=float(row['begin_time']))
            diarized = diarization_module.diarize_audio(tmp.name, chunk_start=float(row['begin_time']))
        Path(tmp.name).unlink(missing_ok=True)

        aligned = assign_speakers_to_segments(asr_result.segments, diarized)
        predicted_speaker = aligned[0].speaker if aligned else 'SPEAKER_UNKNOWN'
        asr_duration = sum(max(0.0, seg.end - seg.start) for seg in asr_result.segments)
        diarized_duration = sum(max(0.0, seg.end - seg.start) for seg in diarized)

        overlap_total = 0.0
        best_overlap = 0.0
        for asr_segment in asr_result.segments:
            segment_best = 0.0
            for diarized_segment in diarized:
                overlap = overlap_duration(
                    asr_segment.start,
                    asr_segment.end,
                    diarized_segment.start,
                    diarized_segment.end,
                )
                overlap_total += overlap
                segment_best = max(segment_best, overlap)
            best_overlap += segment_best

        alignment_coverage = best_overlap / asr_duration if asr_duration > 0 else 0.0
        diarization_to_asr_ratio = diarized_duration / asr_duration if asr_duration > 0 else 0.0

        rows.append({
            'model': model_size,
            'audio_id': row['audio_id'],
            'reference_speaker': row['speaker_id'],
            'predicted_speaker': predicted_speaker,
            'is_unknown': predicted_speaker == 'SPEAKER_UNKNOWN',
            'num_asr_segments': len(asr_result.segments),
            'num_diarized_segments': len(diarized),
            'single_diarized_speaker': len({seg.speaker for seg in diarized}) == 1 if diarized else False,
            'alignment_coverage': alignment_coverage,
            'strong_alignment': alignment_coverage >= STRONG_COVERAGE_THRESHOLD,
            'diarization_to_asr_ratio': diarization_to_asr_ratio,
        })

    return pd.DataFrame(rows)

In [4]:
base_df = evaluate_model('base')
small_df = evaluate_model('small')
results = pd.concat([base_df, small_df], ignore_index=True)

summary = (
    results.groupby('model')
    .agg(
        unknown_rate=('is_unknown', 'mean'),
        single_speaker_rate=('single_diarized_speaker', 'mean'),
        mean_alignment_coverage=('alignment_coverage', 'mean'),
        median_alignment_coverage=('alignment_coverage', 'median'),
        strong_alignment_rate=('strong_alignment', 'mean'),
        mean_diarization_to_asr_ratio=('diarization_to_asr_ratio', 'mean'),
        avg_asr_segments=('num_asr_segments', 'mean'),
        avg_diarized_segments=('num_diarized_segments', 'mean'),
    )
    .reset_index()
)
summary

/Users/satwik/Documents/GitHub/LiveNoteAI/.venv/lib/python3.11/site-packages/pyannote/audio/pipelines/speaker_verification.py:45: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import (
/Users/satwik/Documents/GitHub/LiveNoteAI/.venv/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)
/Users/satwik/Documents/GitHub/LiveNoteAI/.venv/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Co

,model,unknown_rate,single_speaker_rate,mean_alignment_coverage,median_alignment_coverage,strong_alignment_rate,mean_diarization_to_asr_ratio,avg_asr_segments,avg_diarized_segments
0,base,0.224,0.876,0.647344,0.845925,0.540,0.712283,0.916,1.152
1,small,0.212,0.876,0.673475,0.913978,0.584,0.759088,0.952,1.152


In [5]:
inspection = results[[
    'model',
    'audio_id',
    'predicted_speaker',
    'is_unknown',
    'num_asr_segments',
    'num_diarized_segments',
    'alignment_coverage',
    'diarization_to_asr_ratio',
]].sort_values(['model', 'alignment_coverage', 'is_unknown'], ascending=[True, False, True])
inspection.head(20)

,model,audio_id,predicted_speaker,is_unknown,num_asr_segments,num_diarized_segments,alignment_coverage,diarization_to_asr_ratio
72,base,AMI_TS3003d_H01_MTD011UID_0022694_0023520,SPEAKER_00,False,1,1,1.000000,1.369553
111,base,AMI_EN2002d_H03_MEE073_0015416_0015672,SPEAKER_00,False,1,1,1.000000,1.205433
223,base,AMI_ES2004c_H02_MEE014_0120482_0121957,SPEAKER_00,False,5,1,0.999430,1.007494
216,base,AMI_ES2004a_H01_FEE013_0075743_0077073,SPEAKER_00,False,4,1,0.999368,1.040909
59,base,AMI_TS3003d_H00_MTD009PM_0110185_0111458,SPEAKER_00,False,2,1,0.999333,1.021078
65,base,AMI_IS1009b_H03_FIO089_0010963_0011958,SPEAKER_00,False,3,1,0.999148,1.002311
171,base,AMI_TS3003c_H00_MTD009PM_0006622_0007580,SPEAKER_00,False,1,1,0.999106,1.050844
124,base,AMI_ES2004c_H02_MEE014_0072299_0073132,SPEAKER_00,False,3,1,0.998980,1.010105
85,base,AMI_ES2004b_H01_FEE013_0132038_0132750,SPEAKER_00,False,1,1,0.998787,1.028377
152,base,AMI_EN2002b_H03_MEE073_0100452_0101260,SPEAKER_00,False,1,1,0.998787,1.011399
